# IndicWrite — private BharatGen GPU interface verification

**Experimental, user-authorized private Colab research.** This notebook runs the **real pretrained Param-1-2.9B-Instruct checkpoint** inside a loopback-only Streamlit interface and exercises its UI through local Chromium. It does **not** expose the model publicly or upload model-generated sentences to GitHub.

**Security:** Only run after reviewing BharatGen’s research license and the pinned custom-code audit. Select **Runtime → Change runtime type → T4 GPU**. The ZIP produced at the end includes private model output/screenshots and must not be shared publicly.

The verified IndicXlit converter uses a different Python 3.10 environment and is intentionally not part of this Python 3.12 GPU interface test. This notebook tests *Hindi → BharatGen suggestion* only.

In [ ]:
import os, sys, pathlib, subprocess, json, time, shutil, zipfile
from datetime import datetime, timezone
import torch
assert torch.cuda.is_available(), "Use Runtime > Change runtime type > T4 GPU"
print("GPU:",torch.cuda.get_device_name(0), "kernel Python:",sys.version.split()[0])
ROOT=pathlib.Path("/content/IndicWrite-private-ui")
SOURCE_COMMIT="459e8f64c71239bcfd679696764ce896f5d8ad6d"
if not ROOT.exists():
    subprocess.run(["git","clone","--quiet","https://github.com/Rishikeshsanin/Hinglish-Smart-Writing-Assistant.git",str(ROOT)],check=True)
subprocess.run(["git","fetch","--quiet","origin",SOURCE_COMMIT],cwd=ROOT,check=True)
subprocess.run(["git","checkout","--detach",SOURCE_COMMIT],cwd=ROOT,check=True)
print("Audited source:",subprocess.check_output(["git","rev-parse","HEAD"],cwd=ROOT,text=True).strip())

In [ ]:
# Install an isolated Python 3.12 CUDA environment, never change Colab's kernel.
subprocess.run([sys.executable,"-m","pip","install","-q","uv>=0.9,<1"],check=True)
uv=shutil.which("uv")
assert uv
subprocess.run([uv,"python","install","3.12"],check=True)
venv=pathlib.Path("/content/indicwrite-private-ui-env")
subprocess.run([uv,"venv","--python","3.12","--seed",str(venv)],check=True)
PY=str(venv/"bin/python")
subprocess.run([PY,"-m","pip","install","torch==2.6.0","--index-url","https://download.pytorch.org/whl/cu124"],check=True)
subprocess.run([PY,"-m","pip","install","-r",str(ROOT/"requirements-private-research.txt"),"playwright==1.56.0"],check=True)
subprocess.run([PY,"-m","playwright","install","chromium"],check=True)
subprocess.run([PY,"-c","import torch, streamlit; assert torch.cuda.is_available(); print(torch.__version__,streamlit.__version__)"],check=True)

## Launch a private, loopback-bound interface

All three consent flags are explicit. No ngrok, public URLs, tunnels, or Hugging Face output uploads. Streamlit runs locally in the Colab VM; Chromium also runs in the VM.

In [ ]:
EVIDENCE=pathlib.Path("/content")/("IndicWrite-private-gpu-ui-"+datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
EVIDENCE.mkdir(parents=True)
env=os.environ.copy()
env.update({
    "INDICWRITE_PRIVATE_BHARATGEN":"1",
    "INDICWRITE_REVIEWED_BHARATGEN_CODE":"1",
    "INDICWRITE_ACKNOWLEDGE_BHARATGEN_LICENSE":"1",
    "PYTHONPATH":str(ROOT/"src"),
    "TOKENIZERS_PARALLELISM":"false",
})
log=(EVIDENCE/"streamlit-server.log").open("w",encoding="utf-8")
server=subprocess.Popen([
    PY,"-m","streamlit","run","research/private_bharatgen_app.py",
    "--server.headless=true","--server.address=127.0.0.1","--server.port=8501",
],cwd=ROOT,env=env,stdout=log,stderr=subprocess.STDOUT)
import urllib.request
ready=False
for attempt in range(90):
    if server.poll() is not None: break
    try:
        with urllib.request.urlopen("http://127.0.0.1:8501/_stcore/health",timeout=2) as reply:
            if reply.status==200: ready=True; break
    except Exception:
        time.sleep(2)
assert ready, "Private Streamlit server unavailable; check streamlit-server.log"
print("Local-only Streamlit health:",ready)

In [ ]:
# REAL checkpoint inference through actual Streamlit widgets.
# A failure generates diagnostics without making a false success claim.
qa={"source_commit":SOURCE_COMMIT,"checkpoint":"bharatgenai/Param-1-2.9B-Instruct",
    "gpu":torch.cuda.get_device_name(0),"real_pretrained_ui_success":False}
try:
    from playwright.sync_api import sync_playwright, expect
    with sync_playwright() as p:
        browser=p.chromium.launch(headless=True,args=["--no-sandbox","--disable-dev-shm-usage"])
        page=browser.new_page(viewport={"width":1280,"height":900},accept_downloads=True)
        page.goto("http://127.0.0.1:8501",wait_until="domcontentloaded",timeout=90000)
        assert "locked" not in page.locator("body").inner_text().lower(), "Private access gate unexpectedly locked"
        source="वह लड़की रोज़ स्कूल जाता है।"
        page.get_by_role("textbox",name="Original Hindi (Devanagari)").fill(source)
        page.get_by_role("button",name="Generate research suggestion").click()
        expect(page.get_by_text("Model suggestion — not automatically applied")).to_be_visible(timeout=360000)
        candidate=page.locator("[data-testid='stCode'] code").first.inner_text().strip()
        assert candidate and any(0x0900 <= ord(c) <= 0x097F for c in candidate)
        expect(page.get_by_role("button",name="Download privately reviewed suggestion")).to_have_count(0)
        page.screenshot(path=str(EVIDENCE/"private-gpu-ui-before-review.png"),full_page=True)
        page.get_by_role("checkbox",name="I checked the original").check()
        with page.expect_download() as download_info:
            page.get_by_role("button",name="Download privately reviewed suggestion").click()
        file=download_info.value
        assert file.path().read_bytes().decode("utf-8")==candidate
        page.get_by_role("button",name="Clear",exact=True).click()
        expect(page.get_by_role("textbox",name="Original Hindi (Devanagari)")).to_have_value("")
        expect(page.get_by_text("Model suggestion — not automatically applied")).to_have_count(0)
        qa.update({"real_pretrained_ui_success":True,
                   "saw_devanagari":True,"review_required_for_download":True,
                   "download_bytes_verified":True,"clear_hides_previous_output":True})
        page.screenshot(path=str(EVIDENCE/"private-gpu-ui-after-clear.png"),full_page=True)
        browser.close()
except Exception as exc:
    qa.update({"failure_type":type(exc).__name__,"failure":str(exc)[:2000]})
    raise
finally:
    (EVIDENCE/"private-gpu-ui-qa.json").write_text(json.dumps(qa,ensure_ascii=False,indent=2),encoding="utf-8")
    print("PRIVATE_GPU_UI_QA",json.dumps(qa,ensure_ascii=False,indent=2))


## Download private evidence

The ZIP includes **private generated text/screenshot** and must stay outside public GitHub, Google Drive sharing, and deployment artifacts until licensing is clarified. It can be uploaded privately to this ChatGPT conversation for debugging. A browser success here verifies one live UI flow—not independent rewriting quality.

In [ ]:
if "server" in globals() and server.poll() is None:
    server.terminate()
    try: server.wait(timeout=12)
    except subprocess.TimeoutExpired: server.kill()
if "log" in globals(): log.close()
archive=pathlib.Path("/content/IndicWrite_Private_GPU_UI_Evidence.zip")
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as bundle:
    for item in EVIDENCE.iterdir():
        if item.is_file(): bundle.write(item,arcname=item.name)
print("PRIVATE EVIDENCE ZIP:",archive)
from google.colab import files
files.download(str(archive))